# RuleGate Stage 2 Kaggle runner

Runs Single Prompting and Full Sequential Prompting with Both on the 18 design cases only. Read `docs/kaggle_execution.md` first.

In [ ]:
RUN_DESIGN_BASELINES = False
RUN_LOCKED_TEST = False
REPO_URL = 'https://github.com/fakedupiece/nlp-project.git'
CODE_COMMIT = ''  # exact 40-character reviewed RuleGate commit
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL_REVISION = ''  # exact 40-character Hugging Face commit
QUANTIZATION = 'none'  # 'none' or explicitly reviewed '4bit'
DATA_ROOT = '/kaggle/input/policy-compliance-requests-ai'
REPO_ROOT = '/kaggle/working/nlp-project'
OUTPUT_DIR = '/kaggle/working/rulegate-design-baselines'
assert not RUN_LOCKED_TEST, 'The 225-case test remains sealed'

In [ ]:
if RUN_DESIGN_BASELINES:
    import re
    import subprocess
    import sys
    from pathlib import Path
    assert re.fullmatch(r'[0-9a-fA-F]{40}', CODE_COMMIT)
    assert re.fullmatch(r'[0-9a-fA-F]{40}', MODEL_REVISION)
    assert QUANTIZATION in {'none', '4bit'}
    assert not Path(REPO_ROOT).exists(), f'Remove existing path first: {REPO_ROOT}'
    subprocess.run(['git', 'clone', REPO_URL, REPO_ROOT], check=True)
    subprocess.run(['git', '-C', REPO_ROOT, 'checkout', '--detach', CODE_COMMIT], check=True)
    requirements = 'requirements-gpu-4bit.txt' if QUANTIZATION == '4bit' else 'requirements-gpu.txt'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_ROOT], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(Path(REPO_ROOT) / requirements)], check=True)

In [ ]:
if RUN_DESIGN_BASELINES:
    import os
    import torch
    from kaggle_secrets import UserSecretsClient
    assert torch.cuda.is_available(), 'Attach a Kaggle GPU before inference'
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    assert hf_token, 'Kaggle secret HF_TOKEN is missing'
    os.environ['HF_TOKEN'] = hf_token
    del hf_token

In [ ]:
if RUN_DESIGN_BASELINES:
    command = [
        sys.executable, '-m', 'rulegate.stage2', 'run-design',
        '--data-root', DATA_ROOT,
        '--output-dir', OUTPUT_DIR,
        '--code-commit', CODE_COMMIT,
        '--model-id', MODEL_ID,
        '--model-revision', MODEL_REVISION,
        '--quantization', QUANTIZATION,
    ]
    try:
        subprocess.run(command, check=True)
    finally:
        os.environ.pop('HF_TOKEN', None)

Download and review the output directory manually. Kaggle does not synchronize results back to GitHub. Never print `HF_TOKEN`, and do not add a locked-test command during Stage 2.